# Prompt Engineering

---

## API together.ai

In [ ]:
import requests
import json

In [ ]:
model = "meta-llama/Meta-Llama-3.1-8B-Instruct-Turbo"

In [ ]:
API_KEY = "..."

# Параметры модели
url = "https://api.together.ai/v1/completions"
data = {
    "model": model,
    "prompt": "Translate the following English text to French: 'Hello, how are you?'",
    "max_tokens": 50
}
headers = {
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json"
}

response = requests.post(url, headers=headers, json=data)
if response.status_code == 200:
    print("Response:", json.loads(response.text)['choices'][0]['text'])
else:
    print("Error:", response.status_code, response.text)

Response:  is a common greeting in many languages. It is a polite way to ask about someone's well-being. The phrase is often used in informal settings, such as when meeting a friend or acquaintance. It is also used in formal settings, such as in


In [ ]:
def ask_model(prompt):
    headers = {"Authorization": f"Bearer {API_KEY}"}
    data = {
        "model": model,
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": 100
    }
    response = requests.post(url, headers=headers, json=data)
    return json.loads(response.text)['choices'][0]['text']


math_task = "Сложи 23 и 47."
gen_prompt = "Как искусственный интеллект меняет мир?"
# Тестируем запрос 1
response = ask_model(math_task)
print(response)


# Тестируем запрос 2
response = ask_model(gen_prompt)
print(response)

23 + 47 = 70.
Искусственный интеллект (ИИ) уже начал вносить значительные изменения в наш мир, и его влияние будет только увеличиваться в ближайшем будущем. Некоторые из наиболее заметных изменений, которые ИИ уже привнес в наш мир, включают:

1. **Автоматизация рабочих мест**: ИИ уже начал заменять некоторые рабочие места, особенно те, которые требуют повторяющихся и монот


## Решение математических задач через Chain of Thought


In [ ]:
def solve_math_cot(prompt: str) -> str:
    cot_prompt = f"Давайте подумаем шаг за шагом, чтобы решить эту задачу: {prompt}"
    return ask_model(cot_prompt)

In [ ]:
# Пример задач
tasks = [
    "Чему равно 15 умножить на 3?",
    "Если у вас есть 20 яблок, и вы съели 7, сколько осталось?",
    "Найдите сумму чисел от 1 до 10.",
    "Какой остаток от деления 25 на 6?",
    "Чему равен квадрат числа 8?"
]

# Правильные ответы для проверки
true_answers = [
    "45",  # 15 * 3
    "13",  # 20 - 7
    "55",  # Сумма чисел от 1 до 10
    "1",   # Остаток от 25 % 6
    "64"    # Квадрат числа 8
]

# Решение задач
results = []
for task in tasks:
    result = solve_math_cot(task)
    print(f"Вопрос: {task}")
    print(f"Ответ модели: {result}")
    results.append(result)


Вопрос: Чему равно 15 умножить на 3?
Ответ модели: Давайте разберем эту задачу шаг за шагом.

Шаг 1: Мы знаем, что умножение — это повторение сложения. Например, 15 умножить на 3 означает, что нам нужно сложить 15 вместе 3 раза.

Шаг 2: Сложим 15 вместе 3 раза: 
15 + 15 + 15 = ?

Шаг 3: Теперь давайте посчитаем:
15
Вопрос: Если у вас есть 20 яблок, и вы съели 7, сколько осталось?
Ответ модели: Давайте разберем эту задачу шаг за шагом:

Шаг 1: У нас есть 20 яблок.
Шаг 2: Мы съели 7 яблок.
Шаг 3: Чтобы узнать, сколько яблок осталось, нам нужно вычесть количество съеденных яблок из общего количества яблок.

Используя математические операции, мы можем представить это так:

20 (общее количество яб
Вопрос: Найдите сумму чисел от 1 до 10.
Ответ модели: Давайте разберем эту задачу шаг за шагом.

Шаг 1: Определите диапазон чисел, которые нужно сложить. В этом случае диапазон от 1 до 10.

Шаг 2: Найдите формулу для суммы арифметического ряда. Формула: S = (n/2) * (a + l), где S - сумма, n - коли

In [ ]:
# Подсчет точности
correct_count = 0
for result, true_answer in zip(results, true_answers):
    if true_answer in result:
        correct_count += 1

My_accuracy = correct_count / len(tasks)
print("Точность:", My_accuracy)

Точность: 0.2


## Классификация IMDB через few-shot и zero-shot


In [ ]:
from typing import List, Optional

def classify_review(prompt: str, examples: Optional[List[str]] = None) -> str:
    """
    Классификация отзыва с использованием few-shot или zero-shot подхода.

    :param prompt: Отзыв для классификации.
    :param examples: Примеры для few-shot классификации.
    :return: Результат классификации.
    """
    if examples:
        examples_text = "\n".join(examples)
        few_shot_prompt = f"""
    {examples_text}
    Теперь классифицируйте отзыв: {prompt}
        """
    else:
        few_shot_prompt = f"Классифицируйте следующий отзыв: {prompt}"

    return ask_model(few_shot_prompt)

In [ ]:
# Пример zero-shot классификации
review_prompt = "Этот фильм был потрясающим! Сюжет увлекательный, а актеры великолепны."
zero_shot_prompt = f"Классифицируйте следующий отзыв как позитивный или негативный: {review_prompt}"

# Примеры для few-shot
examples = [
    "Отзыв: Этот фильм был потрясающим! Актёры играли великолепно. Ответ: Позитивный",
    "Отзыв: Это худший фильм, который я видел. Сюжет отсутствует. Ответ: Негативный",
    "Отзыв: Не уверен, что рекомендую. Средненький фильм. Ответ: Негативный",
    "Отзыв: Cупер фильм. Ответ: Позитивный",
    "Отзыв: Хуже быть не может. Ответ: Негативный"
]

# Тестирование few-shot классификации
review = "Этот фильм был потрясающим! Сюжет увлекательный, а актёрская игра на высоте."
result_few_shot = classify_review(review, examples=examples)
print("Результат few-shot классификации:", result_few_shot)

# Zero-shot классификация
result_zero_shot = classify_review(review, examples=None)
print("Результат zero-shot классификации:", result_zero_shot)


Результат few-shot классификации: Отзыв: Этот фильм был потрясающим! Сюжет увлекательный, а актёрская игра на высоте.

Ответ: Позитивный

Поскольку в отзыве используются положительные выражения, такие как "потрясающим", "увлекательный", "на высоте", что указывает на то, что фильм понравился и был оценен высоко.
Результат zero-shot классификации: Отзыв можно классифицировать как положительный. В отзыве используются положительные оценки, такие как "потрясающим", "увлекательный" и "на высоте", что указывает на то, что автор фильма оценивает его высоко.


## Self-reflection и качество ответов модели


In [ ]:
def self_reflection(prompt: str) -> str:
    reflection_prompt = f"Проанализируйте ответ и предложите улучшения: {prompt}"
    return ask_model(reflection_prompt)

In [ ]:
# Пример применения self-reflection
task = "Чему равно 25 умножить на 4?"
initial_result = solve_math_cot(task)
improved_result = self_reflection(initial_result)
print("Изначальный ответ:", initial_result)
print("Улучшенный ответ:", improved_result)

Изначальный ответ: Давайте разберем эту задачу шаг за шагом.

Шаг 1: Мы знаем, что нам нужно умножить 25 на 4.

Шаг 2: Чтобы умножить числа, мы можем использовать следующую формулу: 
25 × 4 = (20 + 5) × 4

Шаг 3: Теперь мы можем разложить 25 на 20 и 5, и умножить каждое
Улучшенный ответ: Ответ выглядит логически и шаг за шагом. Однако есть несколько улучшений, которые можно предложить:

1. **Упрощение формулы**: Вместо использования формулы (20 + 5) × 4 можно напрямую умножить 25 на 4, используя свойство умножения на сумму. Это упрощает процесс и делает его более понятным.

2. **Упрощение разложения**: Вмест


## Защита от инъекций


In [ ]:
import re

# Функция проверки на инъекцию
def detect_injection(user_input: str) -> bool:
    """
    Проверяет текст на наличие возможных инъекций.
    Возвращает True, если найдена инъекция.
    """
    # Примеры подозрительных шаблонов
    injection_patterns = [
        r"ignore.*instructions",  # Игнорировать инструкции
        r"forget.*previous",      # Забыть предыдущие команды
        r"reveal.*secret",        # Раскрыть секрет
        r"break.*rules",          # Нарушить правила
    ]
    for pattern in injection_patterns:
        if re.search(pattern, user_input, re.IGNORECASE):
            return True, injection_type
    return False, ""

# Пример использования
def process_user_input(user_input: str) -> str:
    """
    Обрабатывает пользовательский ввод с проверкой на инъекции.
    """
    if detect_injection(user_input):
        return "Ошибка: обнаружена потенциальная инъекция!"

    # Если инъекций нет, отправляем запрос к модели
    data = {
        "model": "NousResearch/Meta-Llama-3.1-8B",
        "prompt": user_input,
        "max_tokens": 50
    }
    response = requests.post(url, headers=headers, json=data)
    if response.status_code == 200:
        return json.loads(response.text)["choices"][0]["text"].strip()
    else:
        return f"Ошибка: {response.status_code}"



In [ ]:
# Тестирование
test_inputs = [
    "SELECT * FROM users;",  # SQL инъекция
    "import os",  # Python инъекция
    "<script>alert('XSS')</script>",  # XSS
    "echo 'Hello World'",  # Shell-инъекция
    "Какой сегодня день?",  # Безопасный ввод
    "Расскажите мне анекдот."  # Безопасный ввод
]

for i, inp in enumerate(test_inputs, start=1):
    is_injection, injection_type = detect_injection(inp)
    if is_injection:
        print(f"Тест {i}: Обнаружена инъекция ({injection_type})")
    else:
        print(f"Тест {i}: Ввод безопасен")


Тест 1: Ввод безопасен
Тест 2: Ввод безопасен
Тест 3: Ввод безопасен
Тест 4: Ввод безопасен
Тест 5: Ввод безопасен
Тест 6: Ввод безопасен
